# Contact and RSA defaults: the decision sweep behind D1, D2 and D3

**A decision sweep over D1 (contact primary), D2 (pLDDT contact mask) and D3 (RSA MaxASA table).**

[`foldenv`](https://github.com/cchin29/foldenv) turns a UniProt accession + residue number into that
residue's **structural microenvironment**, read off the AlphaFold model: relative solvent
accessibility (RSA), 3-state secondary structure, a spatial contact count, a pLDDT confidence
value, and an optional protein-language-model embedding.

Several of those fields depend on choices that have no single right answer. *How close is a
"contact"?* *Should low-confidence AlphaFold residues be allowed to count as contact partners?*
*Which reference table normalises absolute solvent accessibility into RSA?* Each choice shifts
the numbers the package reports, so each is written down as a numbered **decision** in
[`foldenv/decisions.yaml`](https://github.com/cchin29/foldenv/blob/main/foldenv/decisions.yaml)
and is overridable per call via `config.load(overrides=...)`.

This notebook is the evidence behind three of those defaults, and the record of how they were
chosen. It is self-contained: it re-runs the sweep end to end against a live `foldenv`, re-derives
every verdict from the output, and needs nothing but the package and a network connection.

| | decision | default | alternatives tested |
|---|---|---|---|
| **P1** | **D2** — pLDDT contact mask | mask partners with pLDDT < **50** | 70, none |
| **P2** | **D1** — contact primary | **Cα ≤ 8 Å** | Cβ ≤ 5 Å |
| **P4** | **D3** — RSA MaxASA table | **Tien 2013 theoretical** | Tien 2013 empirical, Sander & Rost 1994 |

P1 × P2 is a 3 × 2 grid, run on two proteins. P4 is a separate guardrail check against an
experimental crystal structure. (P3, the embedding-model default, is a document-only decision —
see [§10](#p3).)

**Headline: all three defaults survive — and three tempting reasons for them do not.** That neither
test protein has low-confidence regions worth masking, so P1 cannot matter: false on their own pLDDT
distributions ([§7](#p1)). That P2's result is about the representative *atom*: a matched-radius
control puts the effect in the *radius* ([§8b](#p2b)), and the companion SKEMPI study
reverses the atom term outright (§8). That the lowest
MAE against a crystal identifies the best MaxASA table: it is an arithmetic artifact of the
normalisation ([§9a](#p4a)). Each is worked through where it arises, and one open question falls out
of the second.

## 1. Background — the two structural signals

Only two of `foldenv`'s fields are affected by this sweep, and they measure related but distinct
things.

**RSA — relative solvent accessibility.** DSSP computes each residue's absolute solvent-accessible
surface area (ASA, in Å²) by rolling a water-sized probe over the structure. That number is not
comparable between residue types — a tryptophan simply has more surface than a glycine — so it is
divided by a reference **MaxASA**, the accessibility of that residue type when fully exposed:

$$\mathrm{RSA} = \frac{\mathrm{ASA}}{\mathrm{MaxASA}(\text{residue type})}$$

RSA ≈ 0 means buried, RSA ≈ 1 means fully solvent-exposed. **Decision D3 is which MaxASA table to
use** — and the three candidates disagree, because they were derived differently (theoretical
Gly-X-Gly tripeptide extended-state calculations in Tien 2013, empirical maxima observed in a
structure set in the same paper, and the older Sander & Rost 1994 values). Different tables shift
absolute RSA unevenly — mean |ΔRSA| 0.012 against `tien2013_empirical` but 0.047 against
`sander_rost1994`, which moves 17% of residues by more than 0.10 — enough to carry some
across a "buried" cutoff (§9b).

**Contact count — packing.** For a residue, how many other residues sit inside a neighbourhood
sphere. This is the *packing* signal: a residue in the hydrophobic core is surrounded, a residue on
a flexible loop is not. Two things determine it:

- **D1, the contact primary** — which atoms and which radius define the sphere. Cα–Cα within 8 Å is
  the classical coarse-grained definition (it is roughly the residue-level packing shell); Cβ–Cβ
  within 5 Å is the tighter side-chain-proximity definition used in some contact-map work.
- **D2, the pLDDT mask** — AlphaFold reports a per-residue confidence, pLDDT (0–100). Below ~50 the
  backbone position is essentially a guess, and disordered tails are frequently modelled as long
  loops draped across the folded domain. Those loops generate **contacts that do not exist**. D2
  excludes low-confidence residues from being counted as *partners*. (The queried residue's own
  pLDDT is always reported, never masked — masking it would hide the very fact the user needs.)

**RSA and contacts are computed by completely separate paths.** RSA comes out of DSSP and depends
only on the coordinates and the MaxASA table; contacts come out of a KD-tree over selected atoms.
So the P1/P2 sweep, which varies only contact parameters, leaves RSA **bit-identical across all six
configurations** — a property we assert on rather than assume. Equivalently: P1/P2 is entirely a
question about the packing signal, and P4 is entirely a question about the burial signal.

## 2. Requirements

```bash
pip install foldenv          # 0.3.0 or newer
```

Plus Jupyter itself to run the notebook (`pip install jupyter`); the tables below are rendered with
`IPython.display`, which comes with it. The `foldenv` base install pulls `numpy`, `requests`,
`pyyaml` and `biopython`, and nothing here needs more than that — **no scipy, no torch, no
transformers.** The
embedding is switched off throughout (`embedding.model = "none"`), so the multi-gigabyte `[plm]`
extra is not needed.

**External binary — `mkdssp` (DSSP v4).** Secondary structure and RSA come from DSSP, which
`foldenv` shells out to. Without it every entry point used below fails.

```bash
# macOS
brew tap brewsci/bio && brew install brewsci/bio/dssp
# Linux (recommended — no sudo, no build)
conda install -c conda-forge -c bioconda dssp
# Debian/Ubuntu (verify it is v4; apt has historically shipped 2.x/3.x)
apt-get install dssp
```

**Network access.** The first run downloads two AlphaFold models (P62593, P04637) from
[AlphaFold DB](https://alphafold.ebi.ac.uk/) and, for §9, one crystal structure (1BTL) from
[RCSB](https://www.rcsb.org/). Everything is cached under `./.foldenv_cache/` (relocatable via
`FOLDENV_CACHE_DIR` or the `cache.dir` config leaf), so re-runs are offline and fast: `foldenv`'s
L2 disk cache also stores the DSSP result per (protein, MaxASA table), so `mkdssp` runs once per
pair rather than once per configuration.

Runtime on a warm cache is a few seconds; on a cold cache, a minute or two, dominated by the
downloads and the DSSP calls.

In [1]:
import platform, shutil, sys, warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np

import foldenv
from foldenv import analysis, config, validation
from foldenv import context as ctx

CFG_BASE = config.load(overrides={"embedding": {"model": "none"}})
MKDSSP = shutil.which(CFG_BASE["dssp"]["executable"])

print(f"python      {sys.version.split()[0]}  ({platform.platform()})")
# `foldenv.__version__`, not `version("foldenv")`: the latter reads the installed
# distribution's metadata, which drifts from the code under an editable install.
print(f"foldenv     {foldenv.__version__}")
print(f"numpy       {np.__version__}")
print(f"biopython   {version('biopython')}")
_cache = Path(CFG_BASE["cache"]["dir"])
try:                      # show it relative to the notebook when it is the default location
    _cache = _cache.relative_to(Path.cwd())
except ValueError:
    pass
print(f"cache dir   {_cache}")
print(f"mkdssp      {MKDSSP or 'NOT FOUND — see the install commands above'}")

python      3.12.13  (macOS-26.6.2-arm64-arm-64bit)
foldenv     0.3.0
numpy       2.5.3
biopython   1.88
cache dir   .foldenv_cache
mkdssp      /opt/homebrew/bin/mkdssp


### The committed outputs

Every cell in this file executed, with **mkdssp 4.6.1** on `PATH`, and every assertion below is
live rather than printed. That includes §9's crystal cross-check, which cannot run without a
working binary: `validation.crystal_crosscheck` calls `run_dssp` directly on the experimental
structure, and `foldenv`'s DSSP disk cache is keyed per AlphaFold accession, so it never covers
the crystal side.

The §9 cells are nevertheless written to detect a missing *or* non-functional binary, say so, and
continue rather than abort. A reader without DSSP but with a warm L2 disk cache still gets §5–§8 in full,
and sees §9's reference values printed instead of asserted — never presented as though they had
been recomputed.

**One caveat for readers on a different DSSP build.** `foldenv`'s own cache key includes the DSSP
executable, because different `mkdssp` builds can return different absolute ASA — and therefore
different RSA. The assertions in §6 and §9 pin RSA-derived values to exact decimals. Contact
counts, percentiles and the Spearman/AUROC columns are DSSP-independent and safe; RSA, the
buried/exposed medians and the §9 crystal figures are not. If those assertions fail on your
machine while the contact columns pass, suspect the DSSP build before suspecting `foldenv`.

## 3. Study design

**Two proteins, three known functional residues each.** Both are proteins whose functionally
important residues are independently known from the literature, so we can ask whether the
structural signal picks them out.

- **TEM-1 β-lactamase — UniProt [P62593](https://www.uniprot.org/uniprotkb/P62593), 286 aa.**
  The class A catalytic triad: S70, K73, E166 in *Ambler* numbering (the field-standard numbering
  for β-lactamases), which maps to UniProt **68, 71, 164**. Ambler numbering carries insertions
  relative to the sequence, so that offset is not derivable from the sequence — `foldenv`'s
  `functional_site_stats` therefore self-checks the residue identity at each position (S, K, E)
  and raises if the mapping is wrong. A silently-off-by-two mapping reporting a neighbouring
  residue is the single most dangerous failure mode in an analysis like this one.
- **TP53 — UniProt [P04637](https://www.uniprot.org/uniprotkb/P04637), 393 aa.**
  Three cancer hotspot residues, already in UniProt numbering: **R175, R248, R273**.
  These are *not* three of a kind, and the difference matters when reading the table below:
  R248 and R273 are **DNA-contact** residues, exposed on the DNA-binding surface; **R175 is not** —
  it is a *conformational* mutant that destabilises the DNA-binding domain fold. R175 is buried;
  the other two are not. "Hotspot" is a mutation-frequency claim from the TP53 mutation databases,
  not a structural one.

These two proteins were chosen to be a contrast, not a sample: an enzyme with a buried catalytic
cleft, and a DNA-binding domain with exposed functional surface residues. Three sites per protein
is a handful, and §11 is explicit about what that does and does not license.

**The grid.** 3 pLDDT masks × 2 contact primaries × 2 proteins = 12 runs. The baseline —
the shipped default — is `ca8/50`.

In [2]:
PROTEINS   = ["P62593", "P04637"]                       # TEM-1, TP53
NAMES      = {"P62593": "TEM-1 (P62593)", "P04637": "TP53 (P04637)"}
MASKS      = [("50", 50.0), ("70", 70.0), ("none", 0.0)]  # 0.0 → nothing masked (pLDDT >= 0)
PRIMARIES  = [("ca8", "ca"), ("cb5", "cb")]
CONFIGS    = [f"{p}/{m}" for m, _ in MASKS for p, _ in PRIMARIES]
BASELINE   = "ca8/50"

# The functional-site sets ship with the package (analysis.KNOWN_FUNCTIONAL_SITES).
for uid in PROTEINS:
    print(uid, analysis.KNOWN_FUNCTIONAL_SITES[uid])
print("configs:", CONFIGS)

P62593 {68: 'S', 71: 'K', 164: 'E'}
P04637 {175: 'R', 248: 'R', 273: 'R'}
configs: ['ca8/50', 'cb5/50', 'ca8/70', 'cb5/70', 'ca8/none', 'cb5/none']


## 4. The four metrics

No single number decides this, and pretending otherwise at n = 3 sites per protein would be
dishonest. Four readouts are reported side by side, in descending order of how much weight they
carry.

**(1) Descriptive percentile — the primary readout.** For each functional site, its `contact_count`
and where that sits in the protein's own distribution (the fraction of residues with ≤ that many
contacts). A percentile is the right unit because absolute contact counts are not comparable across
configurations — Cβ-5 Å produces smaller numbers than Cα-8 Å by construction, and comparing 11 to 3
is meaningless while comparing "67th percentile" to "94th percentile" is not. At n = 3 this table
is the candid readout, and everything after it is corroboration.

**(2) Physical-sanity gate — a hard filter, not a score.** Independently of any functional-site
labels, a residue-level packing count has to behave like one. Residues in the **buried core**
(RSA < 0.20) should have a median around **10–20** contacts; residues on the **surface**
(RSA > 0.25) around **4–8**. These bands come from the geometry of protein packing, not from this
dataset. A configuration whose core median falls below ~10 is not measuring burial any more,
whatever it does to the AUROC — so this gate is applied *before* the discrimination numbers, and it
can disqualify a configuration on its own.

**(3) Discrimination — directional only.** AUROC of `contact_count` → functional-vs-everything-else,
and Cliff's δ of the functional sites against the buried core. With 3 positives per protein the
AUROC has a standard error of roughly ±0.15, so differences below ~0.2 are not interpretable. It is
reported to show *direction* and, crucially, whether the direction is **consistent across the two
proteins** — an inconsistency is informative even when neither value alone is.

Cliff's δ = P(a > b) − P(a < b) ∈ [−1, 1] answers a different question than the AUROC: it compares
functional sites against the *buried core* rather than against the bulk, i.e. "are these residues
packed even by core standards?" For a catalytic cleft the expected answer is *no* (δ < 0) — active
sites sit in cavities, which are less packed than solid core.

**(4) Rescale vs reorder — Spearman ρ against the baseline.** Per-residue `contact_count` under a
candidate configuration, rank-correlated against the same vector under `ca8/50`. This separates two
very different kinds of change. ρ ≈ 1 means the configuration is a **rescale**: the same residues in
the same order, on a different numeric scale — downstream consumers that rank or threshold residues
are unaffected. ρ well below 1 means a **reorder**: the configuration disagrees about which residues
are more buried, and swapping it in changes conclusions, not just units.

### Spearman without scipy

**scipy is not a `foldenv` dependency**, and pulling it in for one function would mean this
notebook could not be run by someone who had merely `pip install foldenv`. Spearman's ρ is Pearson's r on the ranks, and the only subtlety is tie
handling: `contact_count` is a small integer with many ties, so ranks must be **midranks** (tied
values all take the mean of the ranks they span) or the correlation is wrong. That is a few lines
of numpy, reused for the AUROC, which needs exactly the same midranks to be a proper
tie-corrected Mann–Whitney statistic.

§6 checks these values against reference numbers **formatted to three decimals**, which bounds
agreement at ~5e-4 and no better. That is enough to confirm the implementation is the same one, and
it is not evidence of floating-point equivalence. No comparison against `scipy.stats.spearmanr` is
claimed: scipy is not a dependency, this notebook deliberately avoids adding one, and a figure
that cannot be reproduced from what ships here would not be worth quoting.

In [3]:
def midranks(x) -> np.ndarray:
    """1-based ranks of `x`, with tied values sharing the mean of the ranks they span."""
    x = np.asarray(x, dtype=float)
    order = x.argsort(kind="mergesort")
    ranks = np.empty(len(x), dtype=float)
    ranks[order] = np.arange(1, len(x) + 1)
    _, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
    sums = np.zeros(len(counts))
    np.add.at(sums, inv, ranks)          # total rank mass per distinct value
    return (sums / counts)[inv]


def spearman(a, b) -> float:
    """Spearman rho = Pearson r on midranks, with ties averaged."""
    ra, rb = midranks(a), midranks(b)
    ra = ra - ra.mean()
    rb = rb - rb.mean()
    return float(ra @ rb / np.sqrt((ra @ ra) * (rb @ rb)))


def auroc(scores, labels) -> float:
    """AUROC via the Mann-Whitney rank formula; higher score => more likely positive.

    Midranks give the correct value under ties. NaN if either class is empty.
    """
    scores, labels = np.asarray(scores, float), np.asarray(labels)
    pos, neg = labels == 1, labels == 0
    n_pos, n_neg = int(pos.sum()), int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = midranks(scores)
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def cliffs_delta(a, b) -> float:
    """Cliff's delta = P(a > b) - P(a < b) in [-1, 1]. NaN if either group is empty."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    return float(np.sign(a[:, None] - b[None, :]).mean())

In [4]:
# Self-checks on values that can be worked out by hand.
assert list(midranks([10, 20, 20, 30])) == [1.0, 2.5, 2.5, 4.0]      # midranks on a tie
assert abs(spearman([1, 2, 3, 4], [10, 20, 30, 40]) - 1.0) < 1e-12   # monotone -> +1
assert abs(spearman([1, 2, 3, 4], [40, 30, 20, 10]) + 1.0) < 1e-12   # anti-monotone -> -1
assert auroc([1, 2, 3, 4], [0, 0, 1, 1]) == 1.0                      # perfectly separated
assert auroc([1, 2, 3, 4], [1, 1, 0, 0]) == 0.0                      # perfectly anti-separated
assert auroc([1, 1, 1, 1], [0, 0, 1, 1]) == 0.5                      # all tied -> chance
assert cliffs_delta([3, 4], [1, 2]) == 1.0
print("metric helpers OK")

metric helpers OK


### The decision rule

> **Change a default only if an alternative improves separation *consistently on both proteins*.
> A gain on one protein is noise.**

This is fixed *before* looking at the results, and it is deliberately conservative. With two
proteins and three sites each, an alternative that wins on one protein and loses on the other is
exactly what sampling noise produces; adopting it would be fitting the default to n = 1. The
asymmetry is intentional — the incumbent default is not neutral ground, it is already documented,
already used, and already reflected in whatever results people have computed with the package.
The burden of proof sits on the challenger.

The physical-sanity gate (metric 2) is a separate, prior filter: a configuration that fails it is
out regardless of what the discrimination numbers say.

## 5. Running the sweep

`structural_profile(uid, cfg)` returns `{position: {aa, rsa, ss3, contact_count, plddt}}` for every
residue, reusing the cached structure, DSSP result and contact KD-tree — so the twelve runs below
cost one DSSP call per protein, not twelve.

In [5]:
def run_config(uniprot_id: str, mask_val: float, primary: str) -> dict:
    """One cell of the P1 x P2 grid: the whole-protein profile plus the four metrics."""
    cfg = config.load(overrides={
        "embedding": {"model": "none"},          # structural fields only, no PLM
        "contacts":  {"primary": primary},       # D1
        "plddt":     {"mask_below": mask_val},   # D2
    })

    profile   = ctx.structural_profile(uniprot_id, cfg)
    positions = sorted(profile)
    cc        = np.array([profile[p]["contact_count"] for p in positions], float)
    rsa       = np.array([np.nan if profile[p]["rsa"] is None else profile[p]["rsa"]
                          for p in positions], float)
    plddt     = np.array([np.nan if profile[p]["plddt"] is None else profile[p]["plddt"]
                          for p in positions], float)

    sites     = analysis.functional_site_stats(uniprot_id, cfg=cfg)  # identity self-checked
    func_pos  = {s.position for s in sites}

    # (2) physical-sanity gate: contact bands for the buried core vs the surface
    core_cc = cc[rsa < cfg["rsa"]["buried_threshold"]]    # RSA < 0.20
    surf_cc = cc[rsa > cfg["rsa"]["exposed_threshold"]]   # RSA > 0.25

    # (3) discrimination, directional at n=3
    labels = np.array([1 if p in func_pos else 0 for p in positions])

    return {
        "uniprot_id": uniprot_id, "mask": mask_val, "primary": primary,
        "positions": positions, "contact_count": cc, "rsa": rsa, "plddt": plddt,
        "sites": [{"position": s.position, "aa": s.aa, "rsa": s.rsa,
                   "rsa_percentile": s.rsa_percentile,
                   "contact_count": s.contact_count,
                   "contact_percentile": s.contact_percentile,
                   "buried": s.buried} for s in sites],
        "sanity": {
            "n_residues":     len(positions),
            "core_median_cc": float(np.median(core_cc)) if len(core_cc) else float("nan"),
            "surf_median_cc": float(np.median(surf_cc)) if len(surf_cc) else float("nan"),
            "core_n": int(len(core_cc)), "surf_n": int(len(surf_cc)),
        },
        "discrimination": {
            "auroc_contact_vs_bulk":     auroc(cc, labels),
            "cliffs_delta_func_vs_core": cliffs_delta(cc[labels == 1], core_cc),
        },
    }


results = {uid: {} for uid in PROTEINS}
for uid in PROTEINS:
    for mlabel, mval in MASKS:
        for plabel, primary in PRIMARIES:
            results[uid][f"{plabel}/{mlabel}"] = run_config(uid, mval, primary)

    # (4) rescale-vs-reorder: rank-correlate each config's per-residue contact vector
    #     against the baseline. Same protein => same positions in the same order.
    base = results[uid][BASELINE]
    for key, res in results[uid].items():
        assert res["positions"] == base["positions"], "position vectors must align"
        res["spearman_vs_baseline"] = spearman(base["contact_count"], res["contact_count"])

    n = base["sanity"]["n_residues"]
    print(f"{NAMES[uid]}: {len(results[uid])} configs x {n} residues")

TEM-1 (P62593): 6 configs x 286 residues


TP53 (P04637): 6 configs x 393 residues


In [6]:
# RSA is DSSP-derived and depends only on the (fixed) MaxASA table, so it must be identical
# across all six contact configurations. Asserted rather than assumed -- if this ever fails,
# a contact knob is leaking into the burial signal.
for uid in PROTEINS:
    base_rsa = results[uid][BASELINE]["rsa"]
    for key, res in results[uid].items():
        assert np.allclose(res["rsa"], base_rsa, equal_nan=True), f"RSA moved in {uid} {key}"
print("RSA is invariant across all P1/P2 configurations, on both proteins")

RSA is invariant across all P1/P2 configurations, on both proteins


In [7]:
from IPython.display import Markdown, display

def render(uid: str) -> Markdown:
    res0      = results[uid][BASELINE]
    site_ids  = [s["position"] for s in res0["sites"]]
    aa        = {s["position"]: s["aa"] for s in res0["sites"]}
    head      = [f"{aa[p]}{p}" for p in site_ids] + \
                ["core_med", "surf_med", "AUROC", "d(func-core)", "rho_vs_base"]
    lines = [f"**{NAMES[uid]}** — functional-site `contact_count` (percentile) by config",
             "", "| config | " + " | ".join(head) + " |",
             "|" + "---|" * (len(head) + 1)]
    for key in CONFIGS:
        r    = results[uid][key]
        byp  = {s["position"]: s for s in r["sites"]}
        san, disc = r["sanity"], r["discrimination"]
        cells = [f"{byp[p]['contact_count']} ({byp[p]['contact_percentile']:.2f})" for p in site_ids]
        cells += [f"{san['core_median_cc']:.0f}", f"{san['surf_median_cc']:.0f}",
                  f"{disc['auroc_contact_vs_bulk']:.2f}",
                  f"{disc['cliffs_delta_func_vs_core']:+.2f}",
                  f"{r['spearman_vs_baseline']:.3f}"]
        mark = " **(default)**" if key == BASELINE else ""
        lines.append(f"| `{key}`{mark} | " + " | ".join(cells) + " |")
    lines += ["", "RSA / burial (config-invariant): " + ", ".join(
        f"**{aa[s['position']]}{s['position']}** rsa={s['rsa']:.3f} "
        f"(pctile {s['rsa_percentile']:.2f}, {'buried' if s['buried'] else 'exposed'})"
        for s in res0["sites"])]
    return Markdown("\n".join(lines))

for uid in PROTEINS:
    display(render(uid))

**TEM-1 (P62593)** — functional-site `contact_count` (percentile) by config

| config | S68 | K71 | E164 | core_med | surf_med | AUROC | d(func-core) | rho_vs_base |
|---|---|---|---|---|---|---|---|---|
| `ca8/50` **(default)** | 11 (0.67) | 11 (0.67) | 11 (0.67) | 12 | 7 | 0.61 | -0.38 | 1.000 |
| `cb5/50` | 3 (0.94) | 1 (0.58) | 2 (0.83) | 2 | 0 | 0.68 | +0.17 | 0.460 |
| `ca8/70` | 11 (0.67) | 11 (0.67) | 11 (0.67) | 12 | 7 | 0.61 | -0.37 | 0.998 |
| `cb5/70` | 3 (0.94) | 1 (0.58) | 2 (0.84) | 2 | 0 | 0.69 | +0.17 | 0.467 |
| `ca8/none` | 11 (0.67) | 11 (0.67) | 11 (0.67) | 12 | 7 | 0.61 | -0.38 | 1.000 |
| `cb5/none` | 3 (0.94) | 1 (0.58) | 2 (0.83) | 2 | 0 | 0.68 | +0.17 | 0.460 |

RSA / burial (config-invariant): **S68** rsa=0.052 (pctile 0.33, buried), **K71** rsa=0.000 (pctile 0.18, buried), **E164** rsa=0.009 (pctile 0.23, buried)

**TP53 (P04637)** — functional-site `contact_count` (percentile) by config

| config | R175 | R248 | R273 | core_med | surf_med | AUROC | d(func-core) | rho_vs_base |
|---|---|---|---|---|---|---|---|---|
| `ca8/50` **(default)** | 14 (0.97) | 7 (0.58) | 12 (0.91) | 12 | 4 | 0.79 | -0.08 | 1.000 |
| `cb5/50` | 3 (0.99) | 0 (0.61) | 0 (0.61) | 1 | 0 | 0.53 | -0.25 | 0.602 |
| `ca8/70` | 14 (0.97) | 7 (0.60) | 12 (0.91) | 12 | 2 | 0.80 | -0.08 | 0.974 |
| `cb5/70` | 3 (0.99) | 0 (0.64) | 0 (0.64) | 1 | 0 | 0.54 | -0.25 | 0.630 |
| `ca8/none` | 14 (0.97) | 7 (0.56) | 12 (0.91) | 12 | 5 | 0.79 | -0.08 | 0.952 |
| `cb5/none` | 3 (0.99) | 0 (0.51) | 0 (0.51) | 1 | 0 | 0.49 | -0.25 | 0.356 |

RSA / burial (config-invariant): **R175** rsa=0.022 (pctile 0.08, buried), **R248** rsa=0.719 (pctile 0.68, exposed), **R273** rsa=0.299 (pctile 0.31, exposed)

## 6. Correctness check

> **If these assertions fail, suspect the data before the code.** They pin exact decimals against
> AlphaFold models fetched live, and AlphaFold DB republishes — the entries used here are v6, and
> v1 is already retired. A republish moves RSA and contact counts legitimately. Check the model
> version before assuming a regression in `foldenv`.

The literals below are **reference values**: the numbers this analysis produced when it was first
run, before the code became part of `foldenv`. They are typed in as constants, and every value
they are compared against is computed fresh by the cells above.

That comparison does two things. It pins the analysis — if a future `foldenv` changed how a contact
or an RSA is computed, these assertions fail rather than the tables quietly reporting something
else. And it means the numbers quoted in the prose throughout this notebook are checked on every
execution instead of being copied by hand.

Assertions are on the **formatted** values, so a mismatch reads as a table difference rather than a
float diff. Six of them re-check the baseline `ca8/50` figures for TEM-1 S68 and TP53 R175; the
contact counts and RSA values there also appear in `foldenv`'s README, so they pin the default
configuration to the documented output.

In [8]:
# Reference values from the first run of this analysis, transcribed as literals. See section 6.
# per config: [site cells...], core_med, surf_med, AUROC, cliffs delta, rho
EXPECTED = {
 "P62593": {
   "ca8/50":   (["11 (0.67)", "11 (0.67)", "11 (0.67)"], "12", "7", "0.61", "-0.38", "1.000"),
   "cb5/50":   (["3 (0.94)",  "1 (0.58)",  "2 (0.83)"],  "2",  "0", "0.68", "+0.17", "0.460"),
   "ca8/70":   (["11 (0.67)", "11 (0.67)", "11 (0.67)"], "12", "7", "0.61", "-0.37", "0.998"),
   "cb5/70":   (["3 (0.94)",  "1 (0.58)",  "2 (0.84)"],  "2",  "0", "0.69", "+0.17", "0.467"),
   "ca8/none": (["11 (0.67)", "11 (0.67)", "11 (0.67)"], "12", "7", "0.61", "-0.38", "1.000"),
   "cb5/none": (["3 (0.94)",  "1 (0.58)",  "2 (0.83)"],  "2",  "0", "0.68", "+0.17", "0.460"),
 },
 "P04637": {
   "ca8/50":   (["14 (0.97)", "7 (0.58)", "12 (0.91)"], "12", "4", "0.79", "-0.08", "1.000"),
   "cb5/50":   (["3 (0.99)",  "0 (0.61)", "0 (0.61)"],  "1",  "0", "0.53", "-0.25", "0.602"),
   "ca8/70":   (["14 (0.97)", "7 (0.60)", "12 (0.91)"], "12", "2", "0.80", "-0.08", "0.974"),
   "cb5/70":   (["3 (0.99)",  "0 (0.64)", "0 (0.64)"],  "1",  "0", "0.54", "-0.25", "0.630"),
   "ca8/none": (["14 (0.97)", "7 (0.56)", "12 (0.91)"], "12", "5", "0.79", "-0.08", "0.952"),
   "cb5/none": (["3 (0.99)",  "0 (0.51)", "0 (0.51)"],  "1",  "0", "0.49", "-0.25", "0.356"),
 },
}
# RSA is config-invariant, so it is listed once per protein: {pos: (rsa, rsa_pctile, buried)}
EXPECTED_RSA = {
  "P62593": {68: ("0.052", "0.33", True), 71: ("0.000", "0.18", True), 164: ("0.009", "0.23", True)},
  "P04637": {175: ("0.022", "0.08", True), 248: ("0.719", "0.68", False), 273: ("0.299", "0.31", False)},
}

checked, failures = 0, []
def check(label, got, want):
    global checked
    checked += 1
    if got != want:
        failures.append(f"{label}: got {got!r}, published {want!r}")

for uid in PROTEINS:
    for key in CONFIGS:
        r = results[uid][key]
        want_cells, w_core, w_surf, w_auc, w_delta, w_rho = EXPECTED[uid][key]
        for s, want in zip(r["sites"], want_cells):
            check(f"{uid} {key} {s['aa']}{s['position']}",
                  f"{s['contact_count']} ({s['contact_percentile']:.2f})", want)
        check(f"{uid} {key} core_med", f"{r['sanity']['core_median_cc']:.0f}", w_core)
        check(f"{uid} {key} surf_med", f"{r['sanity']['surf_median_cc']:.0f}", w_surf)
        check(f"{uid} {key} AUROC",    f"{r['discrimination']['auroc_contact_vs_bulk']:.2f}", w_auc)
        check(f"{uid} {key} delta",    f"{r['discrimination']['cliffs_delta_func_vs_core']:+.2f}", w_delta)
        check(f"{uid} {key} rho",      f"{r['spearman_vs_baseline']:.3f}", w_rho)
    for s in results[uid][BASELINE]["sites"]:
        w_rsa, w_pct, w_buried = EXPECTED_RSA[uid][s["position"]]
        check(f"{uid} {s['position']} rsa",      f"{s['rsa']:.3f}", w_rsa)
        check(f"{uid} {s['position']} rsa_pct",  f"{s['rsa_percentile']:.2f}", w_pct)
        check(f"{uid} {s['position']} buried",   s["buried"], w_buried)

# The baseline figures that are load-bearing outside this notebook: the contact counts and RSA
# values here also appear in foldenv's own README. All six re-assert values already covered
# above, deliberately — they pin the *documented* output of the default configuration, so a
# change to it fails loudly here rather than only in a table.
tem1 = {s["position"]: s for s in results["P62593"][BASELINE]["sites"]}
tp53 = {s["position"]: s for s in results["P04637"][BASELINE]["sites"]}
check("documented TEM-1 S68 contacts",   tem1[68]["contact_count"], 11)
check("documented TEM-1 S68 percentile", f"{tem1[68]['contact_percentile']:.2f}", "0.67")
check("documented TEM-1 S68 rsa",        f"{tem1[68]['rsa']:.3f}", "0.052")
check("documented TP53 R175 contacts",   tp53[175]["contact_count"], 14)
check("documented TP53 R175 percentile", f"{tp53[175]['contact_percentile']:.2f}", "0.97")
check("documented TP53 R175 rsa",        f"{tp53[175]['rsa']:.3f}", "0.022")

if failures:
    raise AssertionError(f"{len(failures)} of {checked} checks failed:\n  " + "\n  ".join(failures))
# 114 distinct reference values; the 6 documented-baseline checks above re-assert values already
# covered, so `checked` counts 120 assertions over those 114 values.
print(f"all {checked} assertions passed, covering {checked - 6} distinct reference values "
      f"(6 configurations x 2 proteins, plus the documented baseline figures)")

all 120 assertions passed, covering 114 distinct reference values (6 configurations x 2 proteins, plus the documented baseline figures)


<a id="p1"></a>
## 7. P1 — the pLDDT contact mask (D2)

**Reading the table.** Down the three `ca8` rows — `ca8/50`, `ca8/70`, `ca8/none` — the
functional-site numbers barely move. Every TEM-1 site stays at 11 contacts / 0.67 on all three
masks. TP53's R175 stays at 14 / 0.97 and R273 at 12 / 0.91; only R248, the exposed DNA-contact
arginine, wobbles (0.58 / 0.60 / 0.56). The core median is 12 under every mask on both proteins,
and the AUROC moves by at most 0.01. Whatever the mask does, it does not change what this analysis
concludes about these sites.

**But "immaterial" is not the same as "inactive".** It is tempting to read the flat functional-site
rows as meaning the mask has nothing to bite on — that both proteins are well-folded, so few
partners are low-confidence, and the stricter cutoff would only matter on a disordered protein. That
reading is wrong here, and the table already contradicts it.

It is true of TEM-1. It is **not** true of TP53: that protein's surface median falls **4 → 2** going
from mask 50 to mask 70,
and rises to **5** with no mask at all, while ρ against the baseline drops to 0.974 and 0.952 —
against 0.998 and 1.000 for TEM-1, which really is inert. The mask is visibly biting on TP53.

That movement also puts `ca8/70` **outside** the physical-sanity gate's surface band on TP53: a
surface median of 2 against a required ~4–8. (The gate's hard-fail condition is on the core, which
stays at 12, so this does not disqualify the configuration. It is worth stating precisely because a
summary of "Cα-8 Å is comfortably in band" would be wrong for that one cell: the Cα-8 Å surface
medians are 7/7/7 on TEM-1 and 4/2/5 on TP53.) It is a small point, and it
points the same way as everything else in this section: the mask is doing something on TP53, and
what it does at 70 is not an improvement. Let us quantify why.

In [9]:
rows = ["| protein | n | median pLDDT | frac < 50 | frac < 70 | surface residues | of those, frac < 70 |",
        "|---|---|---|---|---|---|---|"]
for uid in PROTEINS:
    r  = results[uid][BASELINE]
    pl, rsa = r["plddt"], r["rsa"]
    surf = rsa > CFG_BASE["rsa"]["exposed_threshold"]
    rows.append(f"| {NAMES[uid]} | {len(pl)} | {np.nanmedian(pl):.1f} | {np.mean(pl < 50):.3f} "
                f"| {np.mean(pl < 70):.3f} | {int(surf.sum())} | {np.mean(pl[surf] < 70):.3f} |")
display(Markdown("\n".join(rows)))

| protein | n | median pLDDT | frac < 50 | frac < 70 | surface residues | of those, frac < 70 |
|---|---|---|---|---|---|---|
| TEM-1 (P62593) | 286 | 98.7 | 0.010 | 0.084 | 122 | 0.197 |
| TP53 (P04637) | 393 | 91.6 | 0.298 | 0.402 | 283 | 0.558 |

**TP53 is not a well-folded protein.** Thirty percent of its residues sit below pLDDT 50 and forty
percent below 70 — the transactivation domain, the proline-rich region and the C-terminal
regulatory tail are genuinely disordered, and AlphaFold models them as low-confidence ribbon. Only
the central DNA-binding domain is confidently folded. Fifty-six percent of the residues this
analysis classifies as "surface" are below pLDDT 70.

So this sweep **does** test the mask on a protein with substantial disorder — which rules out the
easy justification for keeping 50 ("neither protein has anything to mask"). The statement that
survives is narrower, and more useful:

> The mask changes contact counts materially in **disordered regions** — that is exactly what it is
> for — and immaterially at the **well-ordered functional sites** the tool is used to characterise.
> Since no functional site in this set sits in a low-confidence region, the sweep cannot distinguish
> 50 from 70 *where it matters*, and there is no evidence for moving.

**Verdict — keep `mask_below: 50`.** Under the decision rule there is nothing to adopt: 70 does not
improve separation on either protein, let alone consistently on both. The positive argument for 50
over 70 is a prior, not a result — pLDDT < 50 is AlphaFold's own "very low / likely disordered"
band, so masking there discards contacts that are probably artefacts, while masking at 70 also
discards contacts in merely-imperfect but real structure. The looser threshold is the more
conservative intervention, and the data give no reason to be less conservative.

**What is still untested.** No functional site in this set lies in a disordered region, so the
question "*which mask is right when the residue you are querying is itself in a low-confidence
neighbourhood?*" is genuinely open. That is the case where the choice would matter, and it needs a
protein whose functional residues sit in or beside disordered segments — an
intrinsically-disordered-region interaction motif, for instance. Users working on such proteins
should treat `mask_below` as a knob to check, not a settled default.

<a id="p2"></a>
## 8. P2 — the contact primary (D1)

Compare the `ca8` rows with the `cb5` rows. Two independent findings, either of which is
sufficient on its own.

D1 bundles two choices — the representative atom and the shell radius — and `ca8` against `cb5`
varies both at once. That is the right comparison for a ship / don't-ship decision on the pair, and
the wrong one for attributing a result to either half, so [§8b](#p2b) completes the 2 × 2 and both
findings below are stated as verdicts on the pair rather than on the atom.

**Cβ-5 Å fails the physical-sanity gate**, and §8b locates the cause in the radius: Cβ at 8 Å
passes the gate comfortably, Cα at 5 Å fails it. The buried-core median collapses from **12** contacts
(Cα-8 Å, on both proteins) to **2** on TEM-1 and **1** on TP53; the surface median goes to **0** on
both, under every mask. The required bands are ~10–20 for the core and ~4–8 for the surface. A 5 Å
Cβ–Cβ shell is a *side-chain-proximity* criterion — it captures only very close packing, and at
residue level it is nearly empty. It is a perfectly reasonable definition of something; it is not a
usable burial/packing count, which is what `contact_count` is documented to be. A statistic whose
modal value on the protein surface is zero cannot rank surface residues at all.

**And it shows no consistent discrimination gain.** This is the decision rule doing its work. The
AUROC moves in **opposite directions** on the two proteins:

| protein (at the default mask 50) | Cα-8 Å | Cβ-5 Å | |
|---|---|---|---|
| TEM-1 (P62593) | 0.61 | **0.68** | Cβ looks better |
| TP53 (P04637) | 0.79 | **0.53** | Cβ is at chance |

A one-protein gain paired with a collapse to chance on the other is the textbook shape of noise,
and the rule was written in advance to refuse exactly this. The effect is not an artefact of the
mask: Cβ-5 Å scores 0.68 / 0.69 / 0.68 on TEM-1 and 0.53 / 0.54 / 0.49 on TP53 across the three
masks, so the sign flip holds throughout the grid. (When quoting this contrast, keep the mask fixed
on both sides: TEM-1's 0.68 is a mask-50 row, so it pairs with TP53's **0.53**, not with the 0.49
from TP53's no-mask row. Both appear in the table above and the conclusion is the same either way,
but a mask-matched pair is the honest one to cite.) Note also that the change is a genuine
**reorder**, not a rescale: ρ against the baseline is 0.36–0.63, so switching primaries does not
merely change the units of `contact_count`, it changes which residues are called more packed than
which. Anything downstream that ranks or thresholds residues would give different answers.

**A note on the negative Cliff's δ.** Under `ca8/50` the functional sites score δ = −0.38 (TEM-1)
and −0.08 (TP53) against the buried core — the functional residues are *less* packed than core
residues. This is not a failure of the signal; it is the point the package's `analysis` module
exists to make. A catalytic site is a **cavity** in the core: buried by RSA, but necessarily
under-packed, because something has to fit in it. The structural axis identifies "buried and
slightly hollow", which is complementary to conservation, not a standalone functional-site
classifier.

**Verdict — keep `primary: ca`, Cα ≤ 8 Å.** The *pair* Cβ-5 Å is disqualified on physical sanity
and unadopted under the decision rule, which is what a ship / don't-ship decision on D1 needs. What
this does **not** establish is that Cα is the better representative atom: [§8b](#p2b) and the SKEMPI
note below both point the other way once the radius is held fixed, and neither this notebook nor the
SKEMPI study has tested Cβ-8 Å as a *default*. Cβ-5 Å remains available (`contacts.primary: "cb"`, with `cb_cutoff` configurable)
for callers who want a side-chain-proximity criterion and know that is what they are asking for.

*Corroboration from a larger set, and a caveat it raises.* A companion study on SKEMPI
(n = 4510 mutation records in crystal complexes, cross-chain contacts against curated interface
labels) finds Cα-8 Å at AUROC **0.690** against Cβ-5 Å at **0.600**, confidence intervals
non-overlapping — the same ordering as here, at a sample size where the AUROC means something. That
is why this verdict does not rest on n = 3.

That study also runs the matched-radius control, and decomposes the 0.090 gap: **+0.133** of it is
the wider radius, against **−0.043** for the atom,
which favours **Cβ**. At a matched 8 Å, Cβ scores 0.733 to Cα's 0.690, paired bootstrap excluding
zero. Two things keep that from being an argument to change D1 — it measures *cross-chain* contacts
while `foldenv` ships a monomer-only view (where neither atom beats chance — Cα-8 Å is at
0.458, below it), and its n counts
records rather than distinct residues. But **whether Cβ-8 Å would be a better default than Cα-8 Å is
an open question that neither study has answered**, and it should not be mistaken for one this
notebook has closed.

<a id="p2b"></a>
### 8b. Matched-radius control — atom against radius

§8 compares **Cα ≤ 8 Å** against **Cβ ≤ 5 Å**, which differ in *two* ways at once — the
representative atom *and* the shell radius. Validating that bundled pair is the right target for a
*ship / don't ship* decision on D1, but it leaves the two factors confounded: on §8's evidence
alone neither the sanity-gate failure nor the AUROC sign flip can be attributed to the atom.

The control is cheap: the same profiles at **Cβ ≤ 8 Å** and **Cα ≤ 5 Å** complete the 2 × 2, and
`foldenv` exposes both cutoffs as config, so no new code is needed. These four arms carry no
reference literals of their own — §6's correctness check pins the six shipped configurations, and
is deliberately untouched by this section.

In [10]:
def arm_profile(uniprot_id, primary, cutoff, mask_val=50.0):
    """Core/surface contact medians and functional-site percentiles for one (atom, radius)."""
    ov = {"primary": primary, "ca_cutoff" if primary == "ca" else "cb_cutoff": cutoff}
    cfg = config.load(overrides={"contacts": ov, "plddt": {"mask_below": mask_val},
                                 "embedding": {"model": "none"}})
    # guard against the silent failure of overriding the cutoff the arm does not use
    assert cfg["contacts"]["primary"] == primary
    assert cfg["contacts"]["ca_cutoff" if primary == "ca" else "cb_cutoff"] == cutoff

    prof = ctx.structural_profile(uniprot_id, cfg)
    pos = sorted(prof)
    cc = np.array([prof[p]["contact_count"] for p in pos], float)
    rsa = np.array([np.nan if prof[p]["rsa"] is None else prof[p]["rsa"] for p in pos], float)
    sites = analysis.functional_site_stats(uniprot_id, cfg=cfg)
    return {"cc": cc, "core": cc[rsa < 0.20], "surf": cc[rsa > 0.25],
            "sites": {s.position: s.contact_percentile for s in sites}}


ARMS = [("ca", 8.0, "Ca-8  (D1 default)"), ("cb", 5.0, "Cb-5  (the alternative tested)"),
        ("cb", 8.0, "Cb-8  (Cb at the default radius)"),
        ("ca", 5.0, "Ca-5  (Ca at the alternative radius)")]

rows = ["| protein | arm | core median | surface median | gate (core) | surface band |",
        "|---|---|---|---|---|---|"]
arm_res = {}
for uid in PROTEINS:
    for primary, cut, label in ARMS:
        r = arm_profile(uid, primary, cut)
        arm_res[(uid, primary, cut)] = r
        cm, sm = int(np.median(r["core"])), int(np.median(r["surf"]))
        # The gate is the core band -- the hard-fail condition §4 defines and the assert
        # below enforces. The surface band is advisory: §7 shows `ca8/70` leaving it without
        # being disqualified, so folding both into one column would contradict the prose.
        gate = "pass" if 10 <= cm <= 20 else "**FAIL**"
        surf = "in band" if 4 <= sm <= 8 else "outside"
        rows.append(f"| {NAMES[uid].split()[0]} | `{label}` | {cm} | {sm} | {gate} | {surf} |")
display(Markdown("\n".join(rows)))

# The gate outcome is a property of the radius alone, not of the atom.
for uid in PROTEINS:
    for primary, cut, _ in ARMS:
        cm = int(np.median(arm_res[(uid, primary, cut)]["core"]))
        if cut == 8.0:
            assert 10 <= cm <= 20, f"{uid} {primary}-8 core median {cm} left the band"
        else:
            assert cm < 10, f"{uid} {primary}-5 core median {cm} did not collapse"
print("asserted: both 8 A arms sit in the core band; both 5 A arms collapse below it, "
      "for either atom")

| protein | arm | core median | surface median | gate (core) | surface band |
|---|---|---|---|---|---|
| TEM-1 | `Ca-8  (D1 default)` | 12 | 7 | pass | in band |
| TEM-1 | `Cb-5  (the alternative tested)` | 2 | 0 | **FAIL** | outside |
| TEM-1 | `Cb-8  (Cb at the default radius)` | 13 | 7 | pass | in band |
| TEM-1 | `Ca-5  (Ca at the alternative radius)` | 3 | 2 | **FAIL** | outside |
| TP53 | `Ca-8  (D1 default)` | 12 | 4 | pass | in band |
| TP53 | `Cb-5  (the alternative tested)` | 1 | 0 | **FAIL** | outside |
| TP53 | `Cb-8  (Cb at the default radius)` | 12 | 4 | pass | in band |
| TP53 | `Ca-5  (Ca at the alternative radius)` | 3 | 2 | **FAIL** | outside |

asserted: both 8 A arms sit in the core band; both 5 A arms collapse below it, for either atom


**The gate is governed by the radius, not the atom.** Both 8 Å arms sit inside the required
10–20 core band (Cβ-8 gives 13 and 12, essentially the default's 12 and 12); both 5 Å arms collapse
out of it, Cα-5 as badly as Cβ-5. So §8's first finding reads precisely: *a 5 Å shell is not a
burial measure*, rather than *Cβ is not a burial measure*. Cβ-5 Å is a side-chain-proximity
criterion because of its radius, not its atom.

**This does not change the verdict, and it does change the reason.** D1 ships the pair Cα-8 Å, and
nothing here argues for shipping Cβ-5 Å. But "Cβ is the wrong atom" is not what the data says, and
the larger SKEMPI study makes that sharper still — see the corroboration note in §8.

<a id="p4"></a>
## 9. P4 — the RSA MaxASA table (D3)

A different question, and a different design. P1/P2 asked which contact definition best separates
residues *within* a protein, judged against functional-site labels. P4 asks whether the RSA
normalisation is faithful to **experiment** — so the comparison is not internal, it is
AlphaFold-versus-crystal.

`validation.crystal_crosscheck` compares TEM-1's AlphaFold model against **PDB 1BTL**, the TEM-1
crystal structure. It aligns the two sequences (crystal "author" numbering is Ambler numbering here,
which does *not* match UniProt — so the mapping is built by alignment, never by trusting residue
numbers), runs DSSP on both, and reports secondary-structure agreement, RSA Pearson r, and RSA MAE
over the residues present in both. Repeat that for each MaxASA table; the acceptance gate is
agreement > 0.85.

**What the design can and cannot see — read this before the table.** `crystal_crosscheck`
normalises *both* sides with the same table: it divides the AlphaFold ASA and the crystal ASA by the
same `MaxASA(aa)` before comparing them. Two consequences, and neither is obvious from the output.

Secondary structure is exactly unchanged across tables (DSSP's SS assignment never consults a
MaxASA table), and Pearson r is almost unchanged, because applying the same monotonic rescale to
both sides of a correlation barely moves it. That near-invariance is not a weakness of the test —
it *is* what P4 exists to establish.

That leaves MAE as the only column that visibly moves between tables, which makes it tempting to
read as the discriminating one. **It is not.** Because both sides share the divisor,

```
MAE  =  mean over residues of   |ASA_AF - ASA_xtal| / MaxASA(aa)
```

the table enters **only as a divisor on a numerator it does not affect**. A table with larger
MaxASA values therefore produces a smaller MAE mechanically, whatever its fidelity to experiment.
§9a below measures how much of the observed spread that accounts for.

In [11]:
P4_TABLES = ["tien2013_theoretical", "tien2013_empirical", "sander_rost1994"]
P4_GATE   = 0.85

p4_rows = None       # None = did not run. Only ever assigned a list once the loop completes,
                     # so an empty list cannot be mistaken for a successful zero-row run.
if MKDSSP is None:
    print("mkdssp not found on PATH -- section 9's crystal cross-check cannot run here.\n"
          "It needs a live DSSP on the 1BTL crystal structure, which foldenv computes on\n"
          "demand (validation.crystal_crosscheck calls run_dssp directly, so the L2 DSSP\n"
          "disk cache does not cover the experimental side). Install mkdssp and re-run.")
else:
    _rows = []
    for table in P4_TABLES:
        cfg = config.load(overrides={"rsa": {"max_asa_table": table},
                                     "embedding": {"model": "none"}})
        try:
            r = validation.crystal_crosscheck("P62593", "1BTL", cfg=cfg)   # downloads 1BTL once
        except (FileNotFoundError, RuntimeError) as exc:
            # FileNotFoundError = binary vanished mid-run; RuntimeError = it ran but returned
            # nothing, which is what a broken build does (e.g. a segfaulting conda mkdssp).
            print(f"mkdssp is present but did not produce a usable result: {exc}\n"
                  "Section 9's crystal cross-check cannot run here.")
            _rows = None
            break
        _rows.append({
            "table": table,
            "ss3_agreement": round(r.ss3_agreement, 4),
            "rsa_pearson":   round(r.rsa_pearson, 4),
            "rsa_mae":       round(r.rsa_mae, 4),
            "n_compared":    r.n_compared,
            "passes_gate":   r.ss3_agreement > P4_GATE and r.rsa_pearson > P4_GATE,
        })

    p4_rows = _rows          # assigned only after every table succeeded

if p4_rows:
    lines = ["| table | SS3 agreement | RSA Pearson r | RSA MAE | n | > 0.85 gate |",
             "|---|---|---|---|---|---|"]
    for x in p4_rows:
        mark = " **(default)**" if x["table"] == P4_TABLES[0] else ""
        lines.append(f"| `{x['table']}`{mark} | {x['ss3_agreement']} | {x['rsa_pearson']} "
                     f"| {x['rsa_mae']} | {x['n_compared']} | {'pass' if x['passes_gate'] else 'FAIL'} |")
    display(Markdown("\n".join(lines)))

| table | SS3 agreement | RSA Pearson r | RSA MAE | n | > 0.85 gate |
|---|---|---|---|---|---|
| `tien2013_theoretical` **(default)** | 0.9962 | 0.9841 | 0.0266 | 263 | pass |
| `tien2013_empirical` | 0.9962 | 0.9842 | 0.0279 | 263 | pass |
| `sander_rost1994` | 0.9962 | 0.9841 | 0.0315 | 263 | pass |

In [12]:
# P4 reference values, transcribed as literals (see section 6). Printed rather than asserted
# when mkdssp is unavailable, so they are never mistaken for something this run computed.
EXPECTED_P4 = [
    {"table": "tien2013_theoretical", "ss3_agreement": 0.9962, "rsa_pearson": 0.9841,
     "rsa_mae": 0.0266, "n_compared": 263, "passes_gate": True},
    {"table": "tien2013_empirical",   "ss3_agreement": 0.9962, "rsa_pearson": 0.9842,
     "rsa_mae": 0.0279, "n_compared": 263, "passes_gate": True},
    {"table": "sander_rost1994",      "ss3_agreement": 0.9962, "rsa_pearson": 0.9841,
     "rsa_mae": 0.0315, "n_compared": 263, "passes_gate": True},
]

if not p4_rows:
    print("P4 cross-check did not run -- assertions skipped. The reference values it should\n"
          "reproduce are, per table (SS3 / Pearson r / MAE, n=263):")
    for x in EXPECTED_P4:
        print(f"  {x['table']:<22} {x['ss3_agreement']} / {x['rsa_pearson']} / {x['rsa_mae']}")
else:
    assert p4_rows == EXPECTED_P4, f"P4 drifted:\n got {p4_rows}\n want {EXPECTED_P4}"
    print(f"all {len(EXPECTED_P4)} published P4 rows reproduced exactly")

all 3 published P4 rows reproduced exactly


<a id="p4a"></a>
### 9a. The MaxASA divisor's contribution to the MAE spread

The claim above is arithmetic, so it can be checked without running DSSP at all. If the table
enters only as a divisor, then to first order

```
MAE(table)  ~  MAE(default) x  mean(1/MaxASA over this protein's composition) ratio
```

and the *predicted* ordering should track the measured one. The only input needed is TEM-1's amino
acid composition, which comes from the DSSP already cached for the default table — no crystal, no
second DSSP run.

If the predicted ratios reproduce the reference MAE ratios closely, the MAE column is measuring the
size of the denominators rather than agreement with experiment, and it cannot be used to rank the
tables.

In [13]:
from foldenv.dssp import MAX_ASA_TABLES

# TEM-1's composition, from the DSSP cached under the default table.
try:
    _d = ctx.get_dssp("P62593", CFG_BASE)
    composition = [r.aa for r in _d.values() if r.aa in MAX_ASA_TABLES[P4_TABLES[0]]]
except (FileNotFoundError, RuntimeError) as exc:
    composition = []
    print(f"no cached DSSP for the default table ({type(exc).__name__}) -- 9a skipped")

if composition:
    pub_mae = {r["table"]: r["rsa_mae"] for r in EXPECTED_P4}
    base_scale = base_mae = None
    lines = ["| table | mean MaxASA | mean(1/MaxASA) | predicted MAE ratio | published MAE "
             "| published ratio |", "|---|---|---|---|---|---|"]
    for t in P4_TABLES:
        tab = MAX_ASA_TABLES[t]
        inv = float(np.mean([1.0 / tab[a] for a in composition]))
        avg = float(np.mean([tab[a] for a in composition]))
        if base_scale is None:
            base_scale, base_mae = inv, pub_mae[t]
        mark = " **(default)**" if t == P4_TABLES[0] else ""
        lines.append(f"| `{t}`{mark} | {avg:.1f} A^2 | {inv:.6f} | {inv / base_scale:.4f} "
                     f"| {pub_mae[t]:.4f} | {pub_mae[t] / base_mae:.4f} |")
    display(Markdown("\n".join(lines)))

    worst = max(abs(float(np.mean([1.0 / MAX_ASA_TABLES[t][a] for a in composition])) / base_scale
                    - pub_mae[t] / base_mae) for t in P4_TABLES)
    print(f"n = {len(composition)} residues; largest divergence between the predicted and the "
          f"published ratio: {worst:.4f}")

| table | mean MaxASA | mean(1/MaxASA) | predicted MAE ratio | published MAE | published ratio |
|---|---|---|---|---|---|
| `tien2013_theoretical` **(default)** | 190.6 A^2 | 0.005613 | 1.0000 | 0.0266 | 1.0000 |
| `tien2013_empirical` | 181.6 A^2 | 0.005918 | 1.0543 | 0.0279 | 1.0489 |
| `sander_rost1994` | 160.5 A^2 | 0.006733 | 1.1996 | 0.0315 | 1.1842 |

n = 286 residues; largest divergence between the predicted and the published ratio: 0.0154


### 9b. A crystal-free cross-check

The crystal comparison rescales both sides with the same table, which is precisely why it is
insensitive to the table. The claim underneath the P4 verdict — *"a rescale, not a reorder"* — can
be tested more directly, and entirely on the AlphaFold side, by asking what actually changes when
the table changes: take the same protein and the same coordinates, compute RSA under each table,
and compare.

If the tables are monotonic rescales of one another then **Spearman ρ ≈ 1** (no residue overtakes
another in burial), **secondary structure is identical** (it never touched the table), and the only
real movement is in the absolute values — visible as a non-trivial MAE, and as a small number of
residues crossing the `buried_threshold` of 0.20. That last number is the practically important one,
because "is this residue buried?" is a call downstream code makes.

In [14]:
default_table = P4_TABLES[0]
dssp_by_table = {}
for table in P4_TABLES:
    cfg = config.load(overrides={"rsa": {"max_asa_table": table}, "embedding": {"model": "none"}})
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")          # mkdssp-version probe noise
            dssp_by_table[table] = ctx.get_dssp("P62593", cfg)
    except (FileNotFoundError, RuntimeError) as exc:
        # Absent binary raises FileNotFoundError; a broken one that returns no residues
        # raises RuntimeError. Both mean "no DSSP for this table", so both degrade the same way.
        print(f"  {table}: no DSSP available ({type(exc).__name__}) -- skipped")

lines = [f"| table vs `{default_table}` | SS3 identical | Spearman rho | Pearson r | MAE | "
         "same buried call (RSA<0.20) |", "|---|---|---|---|---|---|"]
if default_table in dssp_by_table:
    ref = dssp_by_table[default_table]
    for table, cur in dssp_by_table.items():
        if table == default_table:
            continue
        common = sorted(set(ref) & set(cur))
        ss_same = all(ref[p].ss3 == cur[p].ss3 for p in common)
        a = np.array([ref[p].rsa for p in common])
        b = np.array([cur[p].rsa for p in common])
        ok = ~(np.isnan(a) | np.isnan(b))
        a, b = a[ok], b[ok]
        lines.append(
            f"| `{table}` (n={len(a)}) | {'yes' if ss_same else 'NO'} | {spearman(a, b):.4f} "
            f"| {np.corrcoef(a, b)[0, 1]:.4f} | {np.mean(np.abs(a - b)):.4f} "
            f"| {np.mean((a < 0.20) == (b < 0.20)):.4f} |")
    display(Markdown("\n".join(lines)))
else:
    print("no DSSP available for the default table -- 9b skipped")

| table vs `tien2013_theoretical` | SS3 identical | Spearman rho | Pearson r | MAE | same buried call (RSA<0.20) |
|---|---|---|---|---|---|
| `tien2013_empirical` (n=286) | yes | 0.9998 | 0.9996 | 0.0124 | 0.9860 |
| `sander_rost1994` (n=286) | yes | 0.9994 | 0.9982 | 0.0474 | 0.9615 |

**Verdict — keep `max_asa_table: tien2013_theoretical`, and treat the choice as
non-discriminating.**

All three tables clear the > 0.85 gate against the crystal. Secondary-structure agreement is
identical across them to four decimals (0.9962 — as it must be, since DSSP's SS assignment never
consults a MaxASA table), and RSA Pearson r moves only in the fourth decimal
(0.9841 / 0.9842 / 0.9841).

**No column in that table ranks the tables, MAE included.** §9a shows the MAE spread is almost
entirely the divisor: the predicted ratios from mean(1/MaxASA) track the measured MAE ratios to
within about a percent. The default has the lowest MAE because it has the largest MaxASA values,
not because it agrees better with the crystal — so "lowest MAE against experiment" is not a reason
to prefer it, however natural that reading looks in the table.

**So what does decide it?** Nothing in the data — and that is the result. The gate is a floor, not
a ranking, and *"all three pass"* is the finding: **D3 is a reporting recalibration, and no
downstream ranking depends on it.** The default is kept because there is no evidence to move it,
which is a weaker and more honest claim than a ranking would be. A tie-break on "among those that
pass, prefer the highest agreement" does not rescue it either: by Pearson r that picks
`tien2013_empirical` on a 0.0001 lead, which is noise rather than a preference.

**What does depend on the choice**, from §9b: cross-table Spearman ρ on the AlphaFold side is
≈ 0.999, and the buried/exposed call at RSA < 0.20 agrees with the default for **98.6%** of
residues under `tien2013_empirical` and **96.2%** under `sander_rost1994` — so between one and
four residues in a hundred change side. So if you compare RSA *values* across
studies, record which table produced them: mean |ΔRSA| is 0.012 against `tien2013_empirical`
and 0.047 against `sander_rost1994`, and the latter moves 17% of residues past 0.10. If you
only rank
or threshold residues within one study, the choice is nearly irrelevant.

<a id="p3"></a>
## 10. P3 — the embedding default (D5): recorded, not reproduced

The fourth investigation asked whether **D5** — the default PLM, Ankh-large — should change. It is
**not reproduced here**: it needs several multi-gigabyte checkpoints (Ankh, Ankh3, ProstT5, SaProt,
ESM2, ESM C), which would make this notebook unrunnable for most readers, and it has nothing to do
with the structural signal the rest of it is about. The decision and the evidence for it are
recorded here in full instead.

**Decision: keep Ankh-large** — inherited, and deliberately not defended on benchmark accuracy:

- **Ankh-large is [MuLAN](https://github.com/GianLMB/mulan)'s default**, and `foldenv` was
  extracted from work with MuLAN. Keeping the
  same encoder means embeddings computed here match that model's setting without a second
  decision. That is the whole reason, and it is a default rather than a recommendation.
- **This package has no basis for ranking encoders**, because it neither trains nor evaluates a
  downstream model. Which encoder is best for a given task is the caller's experiment to run.
- **ESM C 6B** is the heaviest option, cannot run on MPS, and its transformers path loads on no
  measured release — it is registered but currently unusable, not a recommended swap. `esmc_600m`
  is the ESM C route that works.
- The embedding is **auxiliary**. RSA and contacts carry the structural signal, and the agent tool
  omits the raw vector by default, so this key changes less than its prominence suggests.
- **SaProt** is redundant for this tool by construction rather than by measurement — it consumes
  structure that the caller is already handed explicitly.

So: **Ankh-large is the default and ProstT5 the lighter swap**; among the wider encoders that
actually load, `ankh3_xl` and `esm2_3b` are the 2560-d options. Switching is one key —
`embedding.model` — and the dim follows from the registry. A consumer that has *trained* on one
width will need to retrain for another.

**The gate all candidates had to pass** was row alignment, not accuracy: perturb one residue and
confirm the embedding changes *at that row* (argmax at the mutated position) by a wide margin over
the median row. In `tests/test_embedding.py` the gate is asserted for three of the registered encoders —
`ankh` (`test_embedding_alignment_mutation_sensitive`) and `saprot`
(`test_saprot_structure_aware_forward_and_alignment`), each requiring the mutated row to exceed
five times the median row; and `ankh3_large` (`test_ankh3_length_alignment_regression`), which
asserts the weaker property that the mutation localises to its own row. All three run under
`RUN_HEAVY_EMB=1`. The other seven carry no alignment test here,
so treat the gate as established for those three and unverified for the rest. This is the check that matters for a per-residue tool: a model that
silently misaligns rows would corrupt every position-indexed output regardless of representation
quality.

**Why no ΔΔG numbers appear above.** They were measured under a split that leaks between training
and held-out data, so they could not have settled a choice between encoders in the first place —
and they came from a separate training benchmark that ships neither training code nor benchmark
data here, so a reader could not check them either way. `decisions.yaml`'s D5 comment records
the same reasoning next to the key it governs.

## 11. Limits of this study

Stated plainly, because the tables above are small and it would be easy to over-read them.

**n = 3 functional sites per protein.** The AUROC and Cliff's δ columns are **directional only**.
With three positives the AUROC's standard error is on the order of ±0.15, so no difference smaller
than about 0.2 is interpretable, and the values should be read for their *sign and consistency*
across the two proteins rather than their magnitude. The descriptive percentile table is the candid
primary readout — it is what the decision actually rests on. This is why P2's verdict leans on the
physical-sanity gate (which is label-free and does not depend on n) rather than on the AUROC gap.

**Two proteins is a contrast, not a sample.** TEM-1 and TP53 were picked to differ — a compact
enzyme with a buried catalytic cleft, versus a partly-disordered transcription factor with exposed
DNA-contact residues. That makes a consistency requirement meaningful, but it does not make the
result generalise. Two proteins cannot characterise a distribution over proteins.

**The functional-site sets are curated examples, not a benchmark.** They ship with the package as
`analysis.KNOWN_FUNCTIONAL_SITES` for exactly this descriptive purpose. TP53's three sites are not
even one category: R248 and R273 are DNA-contact residues, R175 is a conformational mutant whose
mechanism is structural — which is why it reads as strongly buried and highly packed while the
other two do not. Pooling them into a single "functional" label is a simplification the table
inherits, and one reason its AUROC should not be taken literally.

**P1's verdict is scoped to well-ordered functional sites.** As §7 sets out, the mask *does* change
contact counts materially in disordered regions; what this study shows is that it does not change
them at the ordered sites being characterised. Whether 50 or 70 is right when the queried residue
itself sits in a low-confidence neighbourhood is untested.

**P4 tests one protein against one crystal structure.** TEM-1 versus 1BTL, 263 comparable residues.
The conclusion that the MaxASA choice is a recalibration rather than a re-ranking is well supported
by the mechanism (monotone rescales) and by the cross-table check in §9b, but the *magnitudes* —
r ≈ 0.984, MAE 0.027 — are one protein's numbers.

**Everything here is monomer-only.** `foldenv` operates on single-chain AlphaFold models, so a
contact count is an *intra-chain* count. An interface residue's binding face is solvent-exposed in
the monomer, so these numbers are close to blind to protein–protein interfaces — measured directly
in the SKEMPI follow-up, where monomer intra-chain contacts predict interface membership at AUROC
0.458 [0.438, 0.479] for the shipped Cα-8 Å arm — a CI that excludes 0.5, so mild
anti-prediction rather than chance. That is a property of the monomer framing, not a defect in D1 or D2, and it is
the documented limitation to keep in mind when applying `contact_count` to binding-site questions.

## 12. Summary

| | decision | tested | verdict | resting on |
|---|---|---|---|---|
| **P1** | D2 — pLDDT contact mask | 50 / 70 / none | **keep 50** | no consistent gain from 70; 50 is AlphaFold's own "very low confidence" band, so it is the more conservative mask. Immaterial *at ordered functional sites*; materially active in disordered regions. |
| **P2** | D1 — contact primary | Cα ≤ 8 Å / Cβ ≤ 5 Å | **keep Cα-8 Å** | Cβ-5 Å fails the physical-sanity gate outright (core median 1–2, surface 0) **and** moves the AUROC in opposite directions on the two proteins. **Scope:** this validates the shipped *pair*, not the atom — §8b shows the gate turns on the 5 Å radius, and at a matched 8 Å the SKEMPI study puts Cβ *ahead* of Cα. Whether Cβ-8 Å would be a better default is open. |
| **P4** | D3 — RSA MaxASA table | Tien-theo / Tien-emp / Sander&Rost | **keep Tien-2013 theoretical** | all three clear the > 0.85 crystal gate; SS3 identical and Pearson r identical to 4 d.p. **No column discriminates** — the MAE spread is a MaxASA-divisor artifact (§9a), not fidelity to experiment. Kept for want of any evidence to move it. The table is a reporting recalibration, not a re-ranking. |

No default changes. `decisions.yaml` stands as shipped — but see §8b: D1's *verdict* survives while
its stated *reason* does not, and a Cβ-8 Å default is an untested possibility rather than a
rejected one.

Each of these is overridable per call — nothing here is a hard-coded constant:

```python
from foldenv import config, get_structural_context

cfg = config.load(overrides={
    "contacts": {"primary": "cb", "cb_cutoff": 5.0},   # D1
    "plddt":    {"mask_below": 70.0},                  # D2
    "rsa":      {"max_asa_table": "sander_rost1994"},  # D3
    "embedding": {"model": "none"},
})
get_structural_context("P62593", 68, config=cfg)
```

If you report RSA or contact counts computed with `foldenv`, **record the configuration** — several
of these values shift the absolute numbers, which is why they are numbered decisions in the first
place.

### Reproducing this notebook

```bash
pip install foldenv jupyter
# plus mkdssp on PATH (see section 2)
jupyter nbconvert --to notebook --execute --inplace contact_and_rsa_decision_sweep.ipynb
```

Everything runs offline after the first execution: the AlphaFold and RCSB downloads and the DSSP
results are cached under `.foldenv_cache/`. Without `mkdssp`, §5–§8 still run from a warm DSSP
cache; §9 does not (see *The committed outputs* in section 2).

### Provenance

This analysis was first run as a set of standalone scripts, before the code it exercises became
`foldenv`. This notebook is now the record of it: the reference values in §6 and §9 are the numbers
that first run produced, typed in here as literals and checked against a live re-run on every
execution. Nothing else is carried over — the notebook needs only `foldenv`, `mkdssp` and a network
connection.

Three of the verdicts rest on narrower grounds than their headline suggests, each argued where it
appears: P1's basis (§7), P2's attribution to the radius rather than the atom (§8b), and P4's
refusal to read MAE as a ranking column (§9a).